#### Improved AIMO3 Qwen3.5-27B with TIR + Agent-Level Optimizations

Key improvements over baseline:
1. **Diverse Prompt Strategies** - Multiple specialized system prompts (code-first, algebraic, brute-force) across attempts
2. **Adaptive Temperature Scheduling** - Vary temperature across attempts for diversity (0.6→1.0→1.2)
3. **Code-Verification Pass** - Spawn a verification attempt that tests top candidate answers with code
4. **Confidence-Informed Self-Consistency (CISC)** - Weight votes by code-success rate + entropy
5. **Speculative Decoding Enabled** - MTP-based spec decode for ~1.5x speed
6. **Smarter Budget Allocation** - Adaptive budget per problem based on difficulty signal
7. **Reduced Prompt Overhead** - Leaner system prompts = more tokens for reasoning
8. **Progressive Early-Stop** - Consensus threshold adapts to time pressure

Reference:
- https://www.kaggle.com/code/shelterw/qwen3-5-w-python
- rStar-Math (ICML 2025): Code-augmented CoT + mutual verification 
- CISC (ACL Findings 2025): Confidence-weighted self-consistency
- GenRM (arXiv 2504.01005): Compute-optimal solve vs verify tradeoff

### (I need to provide detailed explanation when permitted)

In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
!sudo ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so

In [3]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony',
    ], check=True)

In [4]:
import warnings
warnings.simplefilter('ignore')
import os
import sys
import subprocess

In [5]:
set_env(
    input_archive='/kaggle/input/notebooks/shelterw/aimo-3-vllm-v16/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2026.2.1-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.16.1rc1.dev136+g7e9149d9a-cp38-abi3-manylinux_2_31_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2026.2.1-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.8-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/transformers-4.57.6-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/protobuf-6.33.5-cp39

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-tuner 1.4.8 requires keras, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.31.0 requires matplotlib>=3.7.1, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is not installed.
umap-learn 0.5.11 requires scikit-learn>=1.6, which is not installed.
sentence-transformers 5.2.0 requires scikit-learn, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
arviz 0.22.0 require

In [6]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import json
import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed, AutoTokenizer
import kaggle_evaluation.aimo_3_inference_server

In [7]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [8]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [9]:
class CFG:
    
    # =========================================================================
    # IMPROVEMENT 1: Multiple diverse system prompts for different attempt roles
    # Inspired by rStar's diverse reasoning actions + RASC's reasoning-aware SC
    # =========================================================================
    
    # Primary prompt: balanced reasoning (used for ~half the attempts)
    system_prompt_balanced = (
        'You are an IMO-level math solver. Solve step-by-step with rigorous reasoning.\n\n'
        'Strategy: Understand → Explore multiple approaches → Execute the best one → Verify.\n'
        'Use \\boxed{} for your final integer answer (0-99999).\n'
        'Always verify by substitution or alternative method before answering.'
    )
    
    # Code-heavy prompt: prioritize computational verification (rStar-Math inspired)
    system_prompt_code_first = (
        'You are a computational math solver. For EVERY problem:\n'
        '1. First write Python code to explore the problem numerically (small cases, patterns)\n'
        '2. Form a conjecture from computational evidence\n'
        '3. Prove or verify analytically\n'
        '4. Write final verification code to confirm your answer\n\n'
        'Use sympy for symbolic math, numpy for numerics. Always print() results.\n'
        'Final answer: integer 0-99999 in \\boxed{}.'
    )
    
    # Brute-force prompt: try exhaustive/constructive approaches
    system_prompt_brute = (
        'You are a math problem solver who excels at constructive and enumerative approaches.\n\n'
        'Strategy: When possible, enumerate cases or use brute-force search with code.\n'
        'For number theory: test small cases, look for patterns, use modular arithmetic.\n'
        'For combinatorics: enumerate, count, verify with code.\n'
        'For algebra: try specific values, use numerical solvers, then confirm symbolically.\n\n'
        'Write Python code aggressively. Final answer: integer 0-99999 in \\boxed{}.'
    )
    
    # Backward-reasoning prompt: work from answer constraints
    system_prompt_backward = (
        'You are an IMO math solver who works backwards from constraints.\n\n'
        'Strategy:\n'
        '1. Note the answer must be an integer 0-99999\n'
        '2. Identify what constraints the answer must satisfy\n'
        '3. Consider what values are possible given the problem structure\n'
        '4. Work backwards to narrow down candidates, then verify forward\n'
        '5. Use code to check candidates\n\n'
        'Final answer in \\boxed{}.'
    )
    
    # Map attempt indices to prompts for diversity
    # With 8 attempts: 3 balanced, 2 code-first, 2 brute, 1 backward
    attempt_prompts = [
        system_prompt_balanced,    # 0
        system_prompt_code_first,  # 1
        system_prompt_brute,       # 2
        system_prompt_balanced,    # 3
        system_prompt_code_first,  # 4
        system_prompt_backward,    # 5
        system_prompt_balanced,    # 6
        system_prompt_brute,       # 7
    ]
    
    # =========================================================================
    # IMPROVEMENT 2: Adaptive temperature per attempt for diversity
    # Lower temp for code-heavy (precision), higher for creative exploration
    # =========================================================================
    attempt_temperatures = [
        0.7,   # 0: balanced, moderate
        0.6,   # 1: code-first, precise
        0.9,   # 2: brute, exploratory
        1.0,   # 3: balanced, diverse
        0.6,   # 4: code-first, precise
        0.8,   # 5: backward, moderate
        1.1,   # 6: balanced, high diversity
        1.0,   # 7: brute, diverse
    ]
    
    tool_prompt = (
        'Execute Python code in a stateful Jupyter notebook.\n'
        'Available: math, numpy, sympy, itertools, collections, mpmath (64-digit precision).\n'
        'Always use print() to display results. Code persists between executions.'
    )
    
    # IMPROVEMENT 3: Leaner preference prompt (saves ~200 tokens per attempt)
    preference_prompt = (
        'Use sympy for exact symbolic computation, numpy for numerical verification. '
        'Combine symbolic and numerical approaches: derive symbolically, verify numerically.'
    )
    
    served_model_name = 'qwen3.5-27b'
    model_path = '/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'
    
    # =========================================================================
    # IMPROVEMENT 4: Enable speculative decoding for speed
    # Qwen3.5 supports native MTP - ~1.3-1.5x speedup at low concurrency
    # =========================================================================
    spec_config = {
        "method": "mtp",
        "num_speculative_tokens": 1  # MTP-1 has best acceptance rate
    }
    use_spec_decode = True
    
    if "glm" in model_path.lower():
        re_pattern = r"<arg_value>(.*?)</arg_value>"
    elif "nemotron" in model_path.lower() or "qwen3.5" in model_path.lower():
        re_pattern = r"<parameter=[^>]+>(.*?)</parameter>"
    else:
        re_pattern = r"<tool_call>(.*?)</tool_call>"

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    
    # =========================================================================
    # IMPROVEMENT 5: Adaptive early stopping
    # Lower threshold under time pressure, higher when budget allows
    # =========================================================================
    early_stop_high = 4      # Full consensus (plenty of time)
    early_stop_low = 3       # Reduced consensus (under time pressure)
    time_pressure_ratio = 0.3  # Switch to low threshold when <30% time remains
    
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0  # Default, overridden per-attempt
    min_p = 0.02
    
    # IMPROVEMENT 6: Lower presence_penalty (1.5 is very aggressive, can hurt math)
    presence_penalty = 1.0

    tools = [
        {
            "type": "function",
            "function": {
                "name": "python",
                "description": tool_prompt,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "code": {
                            "type": "string",
                            "description": "The Python code to execute."
                        }
                    },
                    "required": ["code"]
                },
            },
        },
    ]

In [10]:
set_seed(CFG.seed)

In [11]:
class AIMO3Template:

    def __init__(self):
        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:
        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

    def apply_chat_template_for_chatml(
        self, 
        messages,
        tokenizer
    ) -> list:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=True,
            tools=CFG.tools,
        )

In [12]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        # IMPROVEMENT 7: Pre-import more useful libraries in sandbox
        self.execute(
            'import math\n'
            'import numpy as np\n'
            'import sympy\n'
            'from sympy import *\n'
            'import itertools\n'
            'import collections\n'
            'from collections import Counter, defaultdict\n'
            'from functools import lru_cache, reduce\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:
        clean_lines = []
        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue
            clean_lines.append(clean_frame)
        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time
            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')
                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type in ('execute_result', 'display_data'):
                data = content.get('data', {})
                text_repr = data.get('text/plain', '')
                if text_repr:
                    stdout_parts.append(text_repr)

            elif msg_type == 'error':
                traceback = content.get('traceback', [])
                return f'[ERROR] {self._format_error(traceback)}'

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        output = ''.join(stdout_parts).strip()
        
        # IMPROVEMENT 8: Truncate very long outputs to save context tokens
        max_output_chars = 3000
        if len(output) > max_output_chars:
            half = max_output_chars // 2
            output = output[:half] + f'\n... [output truncated, {len(output)} chars total] ...\n' + output[-half:]
        
        if not output:
            return 'Code executed successfully (no output).'
        return output

    def reset(self):
        if self._client:
            try:
                self._client.execute(
                    '%reset -f\n'
                    'import math\n'
                    'import numpy as np\n'
                    'import sympy\n'
                    'from sympy import *\n'
                    'import itertools\n'
                    'import collections\n'
                    'from collections import Counter, defaultdict\n'
                    'from functools import lru_cache, reduce\n'
                    'import mpmath\n'
                    'mpmath.mp.dps = 64\n',
                    store_history=False,
                    allow_stdin=False,
                    stop_on_error=False
                )
                time.sleep(0.1)
            except Exception:
                pass

    def close(self):
        if self._client:
            try:
                self._client.stop_channels()
            except Exception:
                pass
        if self._km and self._owns_kernel:
            try:
                self._km.shutdown_kernel(now=True)
            except Exception:
                pass

In [13]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        lines = code.strip().split('\n')
        if not lines:
            return code
        last_line = lines[-1].strip()
        if 'print' in last_line or 'import' in last_line:
            return code
        if not last_line or last_line.startswith('#'):
            return code
        lines[-1] = 'print(' + last_line + ')'
        return '\n'.join(lines)

    @property
    def instruction(self) -> str:
        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')
        if channel:
            message = message.with_channel(channel)
        return message

    def process_sync_plus(self, message: Message) -> list[Message]:
        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

    def process_sync_plus_for_chatml(self, script) -> str:
        self._ensure_session()
        final_script = self._ensure_last_print(script)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return output

In [14]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_path, trust_remote_code=True)
    
        self._preload_model_weights()
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        files_to_load = []
        total_size = 0
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
        cmd = [
            sys.executable, 
            '-m', 'vllm.entrypoints.openai.api_server', 
            '--seed', str(self.cfg.seed), 
            '--model', self.cfg.model_path, 
            '--served-model-name', self.cfg.served_model_name, 
            '--tensor-parallel-size', '1', 
            '--max-num-seqs', str(self.cfg.batch_size), 
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), 
            '--host', '0.0.0.0', 
            '--port', str(self.port), 
            '--dtype', self.cfg.dtype, 
            '--kv-cache-dtype', self.cfg.kv_cache_dtype, 
            '--max-model-len', str(self.cfg.context_tokens), 
            '--stream-interval', str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching',
            '--trust-remote-code',
            '--language-model-only',
        ]
        # IMPROVEMENT: Enable speculative decoding
        if self.cfg.use_spec_decode:
            cmd.extend([
                '--speculative-config',
                json.dumps(self.cfg.spec_config)
            ])
            # MTP requires: disable prefix caching (conflicts with mamba 'align' mode)
            # Remove --enable-prefix-caching if present
            if '--enable-prefix-caching' in cmd:
                cmd.remove('--enable-prefix-caching')
    
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            if return_code is not None:
                self.log_file.flush()
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
        self.sandbox_pool = queue.Queue()
        def _create_sandbox():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass
        return None
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
        if not logprobs_buffer:
            return float('inf')
        total_entropy = 0.0
        token_count = 0
        for top_logprobs_dict in logprobs_buffer:
            if not isinstance(top_logprobs_dict, dict):
                continue
            if not top_logprobs_dict:
                continue
            token_entropy = 0.0
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            total_entropy += token_entropy
            token_count += 1
        if token_count == 0:
            return float('inf')
        return total_entropy / token_count

    def json_loads_code(self, text):
        try:
            json_dict = json.loads(text)
        except:
            json_dict = {}
        return json_dict.get("arguments", {}).get("code", "").strip()
        
    def extract_code_text(self, text: str) -> str | None:
        pattern = re.compile(self.cfg.re_pattern, re.DOTALL)
        matches = pattern.findall(text)
        if matches:
            if "tool_call" in self.cfg.re_pattern:
                return "\n".join(self.json_loads_code(match.strip()) for match in matches)
            return "\n".join(match.strip() for match in matches)
        return None
    
    def _process_attempt_for_chatml(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        temperature: float = None,
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'Code Success Rate': 0.0,
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        logprobs_buffer = []
        
        # IMPROVEMENT: Per-attempt temperature
        temp = temperature or self.cfg.temperature
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
            
            tokenizer = self.tokenizer
            if system_prompt:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": problem}
                ]
            else:
                messages = [
                    {"role": "user", "content": problem}
                ]
            
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = self.template.apply_chat_template_for_chatml(
                    messages, tokenizer
                )
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens:
                    break

                # Build request params
                req_params = dict(
                    model=self.cfg.served_model_name, 
                    temperature=temp, 
                    presence_penalty=self.cfg.presence_penalty,
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={'return_token_ids': True},
                )
                
                # Only enable logprobs when not using spec decode
                if not self.cfg.use_spec_decode:
                    req_params['logprobs'] = self.cfg.top_logprobs
                    req_params['extra_body']['min_p'] = self.cfg.min_p
                
                stream = self.client.completions.create(**req_params)
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
                            if answer is not None:
                                final_answer = answer
                                break
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
                
                messages.append({"role": "assistant", "content": "".join(text_chunks)})
                python_code = self.extract_code_text("".join(text_chunks))
                if python_code:
                    python_calls += 1
                    response_text = local_tool.process_sync_plus_for_chatml(python_code)
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
                    messages.append({"role": "tool", "content": response_text})
                else:
                    answer_text = messages[-1]["content"]
                    final_answer = self._scan_for_answer(answer_text)
                    break
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
        code_success_rate = (python_calls - python_errors) / max(python_calls, 1)
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': final_answer,
            'Code Success Rate': code_success_rate,
        }
    
    # =========================================================================
    # IMPROVEMENT 9: Enhanced answer selection with multi-signal scoring
    # Combines entropy, code success rate, and vote count
    # Inspired by CISC (Confidence-Informed Self-Consistency)
    # =========================================================================
    def _select_answer(self, detailed_results: list) -> int:

        answer_scores = defaultdict(float)
        answer_votes = defaultdict(int)
        
        valid_results = [r for r in detailed_results if r['Answer'] is not None]
        all_entropy_inf = all(r.get('Entropy', float('inf')) == float('inf') for r in valid_results)
        
        for result in valid_results:
            answer = result['Answer']
            entropy = result.get('Entropy', float('inf'))
            code_sr = result.get('Code Success Rate', 0.0)
            python_calls = result.get('Python Calls', 0)
            
            # Base weight from entropy (lower entropy = higher confidence)
            if all_entropy_inf:
                entropy_weight = 1.0
            else:
                entropy_weight = 1.0 / max(entropy, 1e-9)
            
            # Code verification bonus: attempts that used code AND succeeded
            # get a boost (inspired by rStar-Math's code-augmented verification)
            code_bonus = 1.0
            if python_calls > 0:
                code_bonus = 1.0 + 0.5 * code_sr  # Up to 1.5x boost for 100% code success
            
            # Penalize attempts with many errors (unreliable reasoning)
            error_penalty = 1.0
            if result.get('Python Errors', 0) > 2:
                error_penalty = 0.7
            
            weight = entropy_weight * code_bonus * error_penalty
            answer_scores[answer] += weight
            answer_votes[answer] += 1

        scored_answers = []
        for answer, total_score in answer_scores.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_score
            })

        # Sort by votes first (majority), then by score (quality)
        scored_answers.sort(key=lambda x: (x['votes'], x['score']), reverse=True)

        vote_data = [(item['answer'], item['votes'], item['score']) for item in scored_answers]
        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Score'])
        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')
        return final_answer
    
    # =========================================================================
    # IMPROVEMENT 10: Code-based answer verification pass
    # When top answers are close, spawn a quick verification attempt
    # =========================================================================
    def _verify_with_code(self, problem: str, candidates: list[int], deadline: float) -> int | None:
        """Quick code verification of top candidate answers."""
        if time.time() > deadline - 30:  # Need at least 30s
            return None
            
        verify_prompt = (
            f'The following math problem has candidate answers: {candidates}.\n'
            f'Problem: {problem}\n\n'
            'Write Python code to verify which candidate answer is correct. '
            'Test each candidate by substituting back into the problem constraints. '
            'Print the verified answer in \\boxed{}.'
        )
        
        stop_event = threading.Event()
        result = self._process_attempt_for_chatml(
            verify_prompt,
            self.cfg.system_prompt_code_first,
            attempt_index=99,
            stop_event=stop_event,
            deadline=deadline - 5,
            temperature=0.3,  # Low temperature for verification
        )
        
        if result['Answer'] is not None and result['Answer'] in candidates:
            print(f'  Code verification confirmed: {result["Answer"]}')
            return result['Answer']
        return None
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
        
        # IMPROVEMENT: Adaptive early stop based on time pressure
        time_ratio = time_left / self.cfg.notebook_limit
        early_stop = self.cfg.early_stop_high if time_ratio > self.cfg.time_pressure_ratio else self.cfg.early_stop_low
    
        print(f'Budget: {budget:.2f}s | Deadline: {deadline:.2f} | Early stop: {early_stop}\n')
    
        # IMPROVEMENT: Use diverse prompts and temperatures per attempt
        tasks = []
        for attempt_index in range(self.cfg.attempts):
            prompt = self.cfg.attempt_prompts[attempt_index % len(self.cfg.attempt_prompts)]
            temp = self.cfg.attempt_temperatures[attempt_index % len(self.cfg.attempt_temperatures)]
            tasks.append((prompt, attempt_index, temp))
    
        detailed_results = []
        valid_answers = []
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
            for (system_prompt, attempt_index, temp) in tasks:
                future = executor.submit(
                    self._process_attempt_for_chatml, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline,
                    temp,
                )
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    counts = Counter(valid_answers).most_common(1)
                    if counts and counts[0][1] >= early_stop:
                        stop_event.set()
                        for f in futures:
                            f.cancel()
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            results_dataframe['Code Success Rate'] = results_dataframe['Code Success Rate'].round(2)
            display(results_dataframe)
    
        if not valid_answers:
            print('\nResult: 0\n')
            return 0
        
        # IMPROVEMENT 10: If top-2 answers are close in votes, try code verification
        counts = Counter(valid_answers).most_common(3)
        if (len(counts) >= 2 
            and counts[0][1] - counts[1][1] <= 1 
            and time.time() < deadline - 60):
            print('Close vote — running code verification pass...')
            top_candidates = [c[0] for c in counts[:3]]
            verified = self._verify_with_code(problem, top_candidates, deadline)
            if verified is not None:
                print(f'\nVerified Answer: {verified}\n')
                return verified
    
        return self._select_answer(detailed_results)
    
    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'):
            self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass

In [15]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1 into OS Page Cache...
Processed 67 files (111.17 GB) in 138.60 seconds.

Waiting for vLLM server...
Server is ready (took 363.76 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.37 seconds.



In [16]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    final_answer = solver.solve_problem(question_text)
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [17]:
df = pd.read_csv(
    "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv"
).drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

In [18]:
# Reference-set evaluation
#df = pd.read_csv('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv')
#outs = [predict(pl.Series([r['id']]), pl.Series([r['problem']])).to_pandas() for _, r in df.iterrows()]
#merged = df.merge(pd.concat(outs, ignore_index=True), on='id')
#merged['ok'] = merged['answer_x'] == merged['answer_y']
#for _, r in merged.iterrows():
#  print(f"  {'✅' if r['ok'] else '❌'} {r['id']}: got {r['answer_y']:>6}  want {r['answer_x']:>6}")
#print(f"\nAccuracy: {merged['ok'].sum()}/{len(merged)}")

In [19]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # You MUST call this within 15 minutes of the script starting. This is to
    # ensure a "fast fail" in case a bug prevents the inference server from starting.
    # Do anything that might take a long time (like model loading) in the predict
    # function, which has no time limit.
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )


Problem: What is $0\times10$?

Budget: 900.00s | Deadline: 1772655273.53 | Early stop: 4



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer,Code Success Rate
0,5,458,1,0,inf,0,1.0
1,8,508,1,0,inf,0,1.0
2,6,529,1,0,inf,0,1.0
3,3,591,1,0,inf,0,1.0


,Answer,Votes,Score
0,0,4,6.0



Final Answer: 0


Problem: What is $1-1$?

Budget: 900.00s | Deadline: 1772655293.87 | Early stop: 4



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer,Code Success Rate
0,8,468,1,0,inf,0,1.0
1,5,515,1,0,inf,0,1.0
2,2,567,1,0,inf,0,1.0
3,6,660,1,0,inf,0,1.0


,Answer,Votes,Score
0,0,4,6.0



Final Answer: 0


Problem: Solve $4+x=4$ for $x$.

Budget: 900.00s | Deadline: 1772655307.79 | Early stop: 4



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer,Code Success Rate
0,2,516,1,0,inf,0,1.0
1,5,604,1,0,inf,0,1.0
2,3,609,1,1,inf,0,0.0
3,8,611,1,0,inf,0,1.0


,Answer,Votes,Score
0,0,4,5.5



Final Answer: 0

